<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_86_diffusion_deep_dive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🌫️ **Module 86 — Diffusion Deep Dive (SD · FLUX · schedulers · ControlNet · image-LoRA)** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> Phase 12 · Production Extensions


# Module 86 — Diffusion Deep Dive

> M21 was the *primer* — what diffusion is and how a tiny DDPM trains on MNIST. This module is the **production version**: how **Stable Diffusion**, **FLUX**, **SD3** and **DiT-based** models actually work in 2026, the **scheduler zoo** (DDIM, Euler, DPM++, LCM, TCD), **classifier-free guidance**, **ControlNet** + **image-LoRA**, and the **distillation tricks** (LCM, Turbo, Schnell) that get 50-step diffusion down to 1-4 steps. By the end you'll know which model to pick, which scheduler matches it, how to fine-tune a character LoRA, and how to wire ControlNet onto any of them.

### What you'll cover
1. The diffusion math recap (one page)
2. **Latent Diffusion Models** — why SD works in latent space (M85 VAE callback)
3. The **U-Net architecture** that powers SD1.5 / SDXL
4. **Diffusion Transformers (DiT)** — the architecture under SD3 / FLUX / Sora
5. **Schedulers** — DDPM · DDIM · Euler · Heun · DPM++ · UniPC · PNDM · LCM · TCD
6. **Classifier-Free Guidance (CFG)** + the **CFG scale** knob
7. **Conditioning** — text via CLIP / T5, ControlNet, T2I-Adapter, IP-Adapter
8. **Image-LoRA** — Dreambooth, character / style fine-tuning in 10 minutes
9. **Distillation** — LCM · Hyper-SD · SDXL Turbo · FLUX Schnell (50 → 4 → 1 steps)
10. The **2026 model zoo** + production stack (Diffusers · ComfyUI · TensorRT)


## 1 · The diffusion math in one page

Two processes:

**Forward (q):** add Gaussian noise step by step over `T` timesteps. With the cosine / linear `β_t` schedule:

$$x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1 - \bar\alpha_t}\, \epsilon, \qquad \epsilon \sim \mathcal{N}(0, I)$$

At `t=0`, `x_0` is your image; at `t=T`, `x_T` is pure noise.

**Reverse (p_θ):** learn a network `ε_θ(x_t, t, c)` that predicts the noise that was added. With conditioning `c` (text, image, etc.). Loss = simple MSE:

$$\mathcal{L} = \mathbb{E}_{x_0, \epsilon, t}\Big[\|\epsilon - \epsilon_\theta(x_t, t, c)\|^2\Big]$$

At inference, start from `x_T ~ N(0, I)` and iteratively denoise with the **scheduler** (§5) for `K` steps.

### Three equivalent views of the same model
| View | What the network predicts | When you'll meet it |
|---|---|---|
| **ε-prediction** (DDPM, SD1.5) | the noise itself | M21, most SD1.5 code |
| **x₀-prediction** | the clean image directly | some video diffusion models |
| **v-prediction** (Salimans 2022) | `v = √ᾱ ε − √(1−ᾱ) x₀` | SDXL, SD3, FLUX — better for **distillation** |

All three are reparametrisations of the same loss — production models pick `v-prediction` because it has lower variance for distillation.

## 2 · Latent Diffusion Models — why SD works in latent space

Pure-pixel diffusion on a 512×512 RGB image runs 50 U-Net forwards over **786 432 numbers per step**. Slow + expensive.

**LDM (Rombach et al. 2021)** — the architecture that became Stable Diffusion:

```
   image x (512×512×3)
        │
        ▼
   ┌─────────────┐
   │  VAE encoder │ ── E ──►  z (64×64×4)   ← M85 VAE: 32× compression
   └─────────────┘
                            │
                            ▼
                ┌──────────────────────────────┐
                │  DIFFUSION HAPPENS HERE       │
                │  U-Net / DiT on z, not x      │
                │  50 steps × 16 K numbers       │
                └──────────────────────────────┘
                            │
                            ▼
        ┌─────────────┐
        │  VAE decoder │ ── D ──►  x̂ (512×512×3)
        └─────────────┘
```

**The trick.** Compress the image to a `64×64×4` latent (M85 VAE) **once**, run diffusion in that space, decode **once**. **~32× cheaper** than pixel-space diffusion at minimal quality cost.

Every SD-family model — **SD1.5 · SDXL · SD3 · FLUX** — uses this LDM template. The pieces that change between versions are the **VAE** (better encoders), the **denoiser** (U-Net → DiT), the **text encoder** (CLIP → T5), and the **training objective** (ε → v → flow-matching).

## 3 · The U-Net architecture (SD1.5 / SDXL backbone)

The denoiser that won 2020-2023.

```
       z_t (64×64×4)  +  t_emb  +  c_text
              │
              ▼
   ┌──────────────────────────────────────────────┐
   │  ENCODER DOWN-PATH (4 stages)                 │
   │   conv blocks + ResNet + SELF-attention      │
   │                                                │
   │   ↓ stride-2 conv between stages              │
   │   ↓ resolution halves at each step             │
   │   ↓ channels double                            │
   └────────────────────┬──────────────────────────┘
                        │
                        ▼  bottleneck
                ┌────────────────┐
                │  MIDDLE BLOCK  │  ←── CROSS-attention to text c
                │  ResNet + Attn  │
                └────────┬───────┘
                        │
                        ▼
   ┌──────────────────────────────────────────────┐
   │  DECODER UP-PATH (4 stages)                   │
   │   conv blocks + ResNet + SELF-attention      │
   │                          + CROSS-attention   │
   │   ↑ skip-connection from matching encoder    │
   │   ↑ resolution doubles per step              │
   └────────────────────┬──────────────────────────┘
                        │
                        ▼
                  ε̂   (same shape as z_t)
```

Five things every SD U-Net has:
1. **Time embedding** `t_emb` injected at every block — tells the net "how noisy is this?"
2. **Cross-attention** to text embedding `c` injected in the bottleneck + decoder.
3. **Skip connections** between encoder and decoder (the U-Net hallmark).
4. **Self-attention** layers at the lower-resolution stages (16² and 8²) — too expensive at 64².
5. **GroupNorm** instead of BatchNorm (deterministic at inference).

SD1.5: ~860M params. SDXL: ~2.6B params. ControlNet (§7) effectively *clones the encoder half* and injects it back in.

## 4 · Diffusion Transformers (DiT) — what powers SD3 / FLUX / Sora

In 2022 Peebles & Xie asked: **what if we replace the U-Net with a Transformer?** Result: **DiT** — the architecture under SD3, FLUX, Stable Cascade, OpenAI Sora, and Veo.

```
   z_t  ─►  PatchEmbed (2×2 → tokens)
                              │
                              ▼
              ┌──────────────────────────────┐
              │   N × Transformer blocks      │
              │   (self-attn + MLP)           │
              │   conditioned on (t, c)        │
              │   via AdaLN-Zero modulation    │
              └────────┬──────────────────────┘
                       │
                       ▼
              UnpatchEmbed  ─►  ε̂  (same shape as z_t)
```

**Three architectural wins over U-Net:**
1. **Pure transformer scales better with parameters** — DiT-XL/2 outperforms a U-Net of comparable size on ImageNet.
2. **AdaLN-Zero** conditioning — instead of cross-attention, modulate every block's LayerNorm with `(γ, β, α)` predicted from `t` and `c`. Cheaper + better.
3. **No resolution-specific hyperparameters** — easier to scale.

**Variants you'll meet:**
- **DiT** — the original (text-conditional ImageNet).
- **PixArt-α / Σ** — early efficient DiT for text-to-image.
- **SD3 MM-DiT** — Multimodal DiT; processes image and text tokens *together* with full attention between them.
- **FLUX MM-DiT** — Black Forest Labs' 12B-parameter MM-DiT; current open-weights SOTA for text-to-image.
- **Stable Cascade** — a 3-stage DiT pipeline with extreme compression.
- **Sora / Veo / MovieGen / Kling** — DiT scaled to video.

**The 2026 verdict: DiT replaces U-Net for new diffusion models.** SD1.5 / SDXL U-Nets still dominate the *deployed* fine-tune ecosystem because LoRA + ControlNet were trained on them, but every new model from late-2024 onward is DiT-based.

## 5 · Schedulers — the inference-time knob that matters

A scheduler is the **numerical ODE solver** that takes the noise-prediction at each step and produces `x_{t-1}` from `x_t`. Same model, different scheduler → different quality, different number of steps.

| Scheduler | Year | Steps for "OK" | Steps for "great" | Notes |
|---|---|---|---|---|
| **DDPM** | 2020 | ~1000 | ~1000 | the original; almost never used in production |
| **DDIM** | 2020 | 25 | 50 | deterministic, the production default for years |
| **PNDM / PLMS** | 2022 | 25 | 50 | a bit faster than DDIM at equal quality |
| **Euler** / **Euler-a** | 2022 | 20 | 30 | very common in ComfyUI / A1111 |
| **Heun** | 2022 | 20 | 30 | second-order ODE; slightly higher quality |
| **DPM++ 2M / Karras** | 2023 | 15 | 25 | the **A1111 / SDXL community default** in 2024-25 |
| **UniPC** | 2023 | 10 | 20 | predictor-corrector; fast |
| **LCM scheduler** | 2023 | **4** | **8** | for **LCM-distilled** models only |
| **TCD scheduler** | 2024 | **2** | **4** | for **Trajectory-Consistency-Distillation** models |
| **Flow-Matching Euler** | 2024 | 25 | 50 | SD3 / FLUX use this (rectified flow formulation) |

**How to pick.** Without distillation: **DPM++ 2M Karras** (25 steps) is the unbeaten default for SDXL. For FLUX: **Flow-Matching Euler** (25-30 steps). For real-time / mobile: **LCM** (4 steps) or **TCD** (2-4 steps) distilled checkpoints.

In [ ]:
# How a scheduler is used in HuggingFace Diffusers
scheduler_snippet = '''
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler

pipe = StableDiffusionXLPipeline.from_pretrained("stabilityai/stable-diffusion-xl-base-1.0")

# Swap to DPM++ 2M Karras
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config,
    use_karras_sigmas=True,
    algorithm_type="dpmsolver++",
)

image = pipe(
    prompt="a photo of a cat in space, detailed, 4k",
    num_inference_steps=25,        # ← scheduler-specific sweet spot
    guidance_scale=7.5,             # ← CFG (next section)
).images[0]
'''
print(scheduler_snippet)

## 6 · Classifier-Free Guidance (CFG)

How does the model know "make it look more like the prompt"? The CFG trick.

At inference, the model is run **twice**:
- once with the actual text conditioning `c`,
- once with the **empty / null** conditioning `∅`.

The denoiser output is then mixed:

$$\hat\epsilon = \epsilon_\theta(x_t, \emptyset) + s \cdot (\epsilon_\theta(x_t, c) - \epsilon_\theta(x_t, \emptyset))$$

`s` is the **CFG scale** (or *guidance scale*). Production sweet spots:
- **SD1.5**: `s ≈ 7`
- **SDXL**: `s ≈ 6-8`
- **SD3 / FLUX-dev**: `s ≈ 3.5-5`
- **FLUX Schnell / SDXL Turbo / distilled**: `s = 0` (no CFG, baked into the distillation)

Higher `s` → more prompt-faithful but **saturated / over-cooked** images. Lower `s` → more diverse but **less prompt-aligned**. Most artefacts ("plastic skin," "too-saturated colours," "extra fingers") come from **CFG too high**.

## 7 · Conditioning — text, image, sketch, depth, pose

The whole point of "controllable" diffusion is feeding more than just text into the denoiser. Five families to know:

| Mechanism | What it adds | Where |
|---|---|---|
| **CLIP text encoder** | text → 77×768 tokens | SD1.5, SDXL |
| **T5 text encoder** | longer, richer text → tokens | SD3, FLUX, PixArt |
| **CLIP image encoder** | image → tokens (for IP-Adapter, image-prompt) | every era |
| **ControlNet** | trainable encoder clone that injects spatial conditions (depth, canny, pose, scribble, segmentation) into the denoiser | SD1.5, SDXL, SD3, FLUX |
| **T2I-Adapter / IP-Adapter** | lightweight alternatives to ControlNet | SDXL ecosystem |
| **ReferenceOnly / InstantID / PuLID** | identity preservation from a reference photo | character generation |

### ControlNet in one paragraph
**Zhang et al. 2023.** Take a frozen SD U-Net. Clone its **encoder half**. Train *only the clone* to map an extra input (depth map, canny edges, pose skeleton) into hidden states that get **added** to the original encoder's hidden states at every resolution. Result: a conditional model that respects the spatial layout in your conditioning image.

ControlNet-XL versions exist for SDXL. **FLUX.1 ControlNets** (depth, canny, IP-Adapter) shipped in 2024-25. The ControlNet ecosystem is one of the main reasons SD1.5 / SDXL still dominate **production fine-tuning** even as DiT-based models lead in quality.

In [ ]:
# ControlNet usage in 8 lines (HuggingFace Diffusers)
controlnet_snippet = '''
from diffusers import StableDiffusionXLControlNetPipeline, ControlNetModel
from controlnet_aux import OpenposeDetector

cn = ControlNetModel.from_pretrained("diffusers/controlnet-openpose-sdxl-1.0", torch_dtype=torch.float16)
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0", controlnet=cn, torch_dtype=torch.float16).to("cuda")

pose_img = OpenposeDetector.from_pretrained("lllyasviel/Annotators")(reference_image)
out = pipe(prompt="a woman dancing in a city", image=pose_img,
           controlnet_conditioning_scale=0.8, num_inference_steps=25).images[0]
'''
print(controlnet_snippet)

## 8 · Image-LoRA — fine-tuning in 10 minutes

The same LoRA recipe from M39 / M58 (text LLMs) applies to image diffusion U-Nets and DiTs. Three flavours:

| Flavour | What you teach | Typical effort |
|---|---|---|
| **Dreambooth** | a *specific subject* (your face, your dog) | 10-30 images, 1-4 K steps |
| **Textual Inversion** | a new token via embedding learning only | tiny — a couple-MB file |
| **LoRA / LyCORIS** | a *style* or *concept* via low-rank adapter | 30-200 images, ~10 K steps |

The training loop for image-LoRA is the M39 loop minus tokenisation: load base model, attach LoRA to the U-Net's cross-attention (and optionally to text encoder), train denoising loss on your dataset, save the ~30-200 MB LoRA file.

**`kohya-ss/sd-scripts`** is the canonical training framework. **`diffusers.training_utils`** is the HF-native path. **Civitai** is the de-facto registry of community LoRAs (hundreds of thousands of styles + characters as of 2026).

```python
# Loading a community LoRA on SDXL — one line:
pipe.load_lora_weights("civitai/some-character-lora.safetensors", adapter_name="char")
pipe.set_adapters(["char"], adapter_weights=[0.7])
```

**Stack LoRAs** for "character + style + lighting" combinations — `set_adapters(["char", "style", "light"], weights=[0.6, 0.8, 0.4])`. Most production diffusion stacks (Midjourney, Leonardo, Civitai-driven SaaS) are built on this stack-LoRAs pattern.

## 9 · Distillation — 50 steps → 4 → 1

The biggest 2023-25 advance in diffusion was **distillation** — train a small student to match a big teacher's *trajectory* in far fewer steps.

| Method | Steps | Notes |
|---|---|---|
| **Progressive Distillation** (2022) | halved per round | the original recipe; ~4-8 steps |
| **LCM** (Latent Consistency Models, 2023) | **4 steps** | hugely popular SDXL distillation; CFG = 0 baked in |
| **LCM-LoRA** | 4 steps | LoRA on top of any base model that adds LCM behaviour |
| **SDXL Turbo** (2023) | **1-4 steps** | Adversarial Diffusion Distillation (ADD); StabilityAI |
| **Hyper-SD** (2024) | **1-8 steps** | trajectory-segmented consistency; SOTA few-step quality |
| **TCD** (Trajectory Consistency Distillation, 2024) | **2-4 steps** | improves on LCM |
| **FLUX.1 Schnell** (2024) | **4 steps** | the distilled FLUX variant — Apache-2.0, free for commercial |
| **DMD / DMD2** (2024) | **1 step** | distribution matching distillation; competitive 1-step quality |

The 2026 production reality:
- **For research / batch quality**: use the **base model** at 25-30 steps.
- **For interactive UX / mobile**: use a **distilled checkpoint** at 1-4 steps.
- **For "10 ms per image" / real-time**: **DMD / Hyper-SD / FLUX Schnell** at 1-2 steps; pair with **TensorRT** (M71) for 2-3× more.

## 10 · The 2026 model zoo + production stack

### Open-weights model zoo

| Model | Released | Type | License | Niche |
|---|---|---|---|---|
| **SD 1.5** | 2022 | U-Net + CLIP | open, RAIL | LARGEST community / LoRA / ControlNet ecosystem |
| **SD XL** | 2023 | U-Net + 2× CLIP | open, RAIL | production quality, fast inference |
| **SD 3 / 3.5** | 2024 | MM-DiT + T5 | open (community) | text rendering, anatomy |
| **FLUX.1 dev** | 2024 | MM-DiT (12B) + T5 | non-commercial | SOTA open-weights image quality |
| **FLUX.1 Schnell** | 2024 | distilled FLUX dev | **Apache-2.0** | best commercial-friendly open model |
| **FLUX.1 Pro / Ultra** | 2024-25 | hosted API only | commercial | API quality leader |
| **Stable Cascade** | 2024 | 3-stage DiT | open | extreme compression |
| **PixArt-Σ / α** | 2023-24 | DiT + T5 | open | efficient DiT |
| **AuraFlow** | 2024 | flow-matching DiT | open | community alternative |
| **HunyuanDiT / Kolors** | 2024 | bilingual DiT | open | strong CN+EN support |
| **OmniGen / Janus / Lumina-Next** | 2024-25 | unified multimodal | open | image gen + understanding |

### Hosted leaders 2026
- **Midjourney v6/v7** — top aesthetics, closed.
- **DALL-E 3 / GPT-Image** (OpenAI) — strong text rendering, character coherence.
- **Imagen 3 / 4** (Google) — top photorealism.
- **Ideogram 2** — text-in-image leader.
- **FLUX.1 Pro / Ultra** — open-weights leader, hosted via Replicate / Fal / Together.

### Production stack

| Layer | Tool |
|---|---|
| **High-level Python API** | 🤗 **Diffusers** (`pip install diffusers`) |
| **Node-based GUI / workflow** | **ComfyUI** — the production-favourite for complex pipelines |
| **Web UI for users** | **Automatic1111 / Forge / SD.Next** |
| **Training** | **kohya-ss/sd-scripts**, `diffusers.training_utils`, **OneTrainer** |
| **Quantisation** | INT8 / FP8 via **Diffusers' quantization**, **TensorRT** (M71) |
| **Serving** | Diffusers + **vLLM-style batching** (Diffusers' `attention_slicing`, `enable_xformers`), **NVIDIA Triton** (M71) |
| **Mobile / edge** | Core ML, ONNX Runtime, **MLC LLM** for SD on phones; FLUX Schnell on iPhone 15+ at ~3s |
| **Cost / cost optimisation** | LoRA + distillation + INT8; M79 |

```python
# Production-shape SDXL + LoRA + DPM++ 2M Karras in 12 lines
diffusers_snippet = '''
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler
import torch

pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16, variant="fp16",
).to("cuda")
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config, use_karras_sigmas=True)
pipe.load_lora_weights("my-character-lora.safetensors", adapter_name="char")
pipe.set_adapters(["char"], adapter_weights=[0.7])
pipe.enable_xformers_memory_efficient_attention()        # 2x speedup + lower VRAM

image = pipe("a photo of <char> in Paris, cinematic", num_inference_steps=25, guidance_scale=6).images[0]
'''
print(diffusers_snippet)

## ✅ Recap

- Diffusion = noise-prediction over `T` steps; **LDM** runs that in a `64×64×4` **VAE latent** (M85) ≈ 32× cheaper than pixel-space.
- **U-Net** (SD1.5 / SDXL) → **DiT** (SD3 / FLUX / Sora / Veo) is the architectural arc.
- **Schedulers** matter as much as the model — **DPM++ 2M Karras** (25 steps) is the SDXL default; **flow-matching Euler** for FLUX; **LCM / TCD** for distilled 4-step models.
- **CFG** = run with + without conditioning; mix with scale `s`. Sweet spots: SDXL `~6-8`, FLUX `~3.5-5`, distilled `0`.
- **ControlNet / IP-Adapter / T2I-Adapter** for spatial / image conditioning. **Stack LoRAs** for character × style × lighting.
- **Distillation** (LCM / Hyper-SD / Turbo / Schnell / DMD) gets you to 1-4 steps for interactive UX.
- 2026 production stack: **Diffusers** + **ComfyUI** + **kohya** + **TensorRT** + **LoRA registry** (Civitai).

Next: **M87 — GraphRAG + Knowledge Graphs**.
